In [76]:
from pathlib import Path

import torch
import chess
import chess.svg
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
import pandas as pd

from MaskedDiffusion.model import MaskedDiffusion
from tokenization.tokenization import tokens_to_fen, tokens_to_move, tokenize_move, tokenize_fen, tokenize_themes, scale_ratings, unscale_ratings, theme_preprocessor

In [ ]:
model_path = Path(".") / "runs" / "supervised" / "best_move_model" / "model_0980000.pt"
checkpoint = torch.load(model_path, map_location="cpu", weights_only=False)
model = MaskedDiffusion(checkpoint["config"])
model.load_state_dict(checkpoint["model"])

trainset = torch.load(Path(".") / "dataset" / "with_best_move" / "trainset.pt", weights_only=False, map_location="cpu")
testset = torch.load(Path(".") / "dataset" / "with_best_move" / "testset.pt", weights_only=False, map_location="cpu")

<All keys matched successfully>

In [44]:
def get_move(fen: str, themes: list[str], rating: float | int, steps=64, temperature=0.01):
    fen = torch.tensor([tokenize_fen(fen)], dtype=torch.long)
    themes = torch.from_numpy(tokenize_themes([themes])).to(torch.float32)
    rating = scale_ratings(torch.tensor([rating], dtype=torch.float32))
    tokens = model.sample_move(fen, themes, rating, steps=steps, temperature=temperature, return_steps=False)
    move = tokens[0, model.config.fen_length:]
    return tokens_to_move(move)

In [ ]:
n = 8
html = "<table><tr>"

for i in range(n * 4):
    fen, move, themes, rating = testset[i]
    fen_string = tokens_to_fen(fen)
    themes = theme_preprocessor.inverse_transform(themes.unsqueeze(0))[0]
    rating = unscale_ratings(rating)
    predicted_best_move = get_move(fen_string, themes, rating, steps=4, temperature=0.01)
    true_best_move = tokens_to_move(move)
    
    board = chess.Board(fen_string)
    arrows = []
    
    if predicted_best_move == true_best_move:
        arr = chess.svg.Arrow(chess.parse_square(true_best_move[:2]), chess.parse_square(true_best_move[2:4]), color='#28a745')
        arrows.append(arr)
    else:
        true_arr = chess.svg.Arrow(chess.parse_square(true_best_move[:2]), chess.parse_square(true_best_move[2:4]), color='#28a745')
        pred_arr = chess.svg.Arrow(chess.parse_square(predicted_best_move[:2]), chess.parse_square(predicted_best_move[2:4]), color='#dc3545')
        arrows.extend([true_arr, pred_arr])
    
    svg = chess.svg.board(board, arrows=arrows, size=250)
    
    html += f"<td><div style='text-align: center; font-family: sans-serif;'>{svg}<br><b>True:</b> {true_best_move} <br> <b>Pred:</b> {predicted_best_move}</div></td>"
    
    if (i + 1) % 4 == 0 and i < (n * 4) - 1:
        html += "</tr><tr>"

html += "</tr></table>"
display(HTML(html))


In [75]:
# 1. Define interactive input fields
fen_input = widgets.Text(
    value='1k1r4/p1p4r/1p2R3/3P1qpp/2PQ1p2/1PN1n3/P5PP/4R2K b - - 5 25',
    description='FEN:',
    layout=widgets.Layout(width='80%')
)

themes_input = widgets.Text(
    value='crushing,middlegame,short',
    description='Themes:',
    layout=widgets.Layout(width='80%')
)

rating_input = widgets.IntText(
    value=2076,
    description='Rating:'
)

button = widgets.Button(
    description='Predict Move', 
    button_style='info'
)

# Output area where the board will be drawn
output = widgets.Output()

def on_button_click(b):
    with output:
        # Clear the previous visualization
        clear_output(wait=True)
        
        fen_string = fen_input.value
        themes = [t.strip() for t in themes_input.value.split(',') if t.strip()]
        rating = rating_input.value
        
        try:
            # Generate the predicted move using your model
            predicted_best_move = get_move(fen_string, themes, rating, steps=1, temperature=0.01)
            
            # Setup the board
            board = chess.Board(fen_string)
            
            # Only draw the predicted arrow in red, since there isn't a "true" move for arbitrary FENs
            arrows = [
                chess.svg.Arrow(
                    chess.parse_square(predicted_best_move[:2]), 
                    chess.parse_square(predicted_best_move[2:4]), 
                    color='#dc3545' # Red 
                )
            ]
            
            # Draw the SVG
            svg = chess.svg.board(board, arrows=arrows, size=350)
            
            # Show output as HTML
            html = f"<div style='margin-top: 10px; text-align: left; font-family: sans-serif;'>"
            html += f"{svg}<br><br>"
            html += f"<b>Predicted Move:</b> <span style='color: #dc3545;'>{predicted_best_move}</span><br>"
            html += f"<b>Themes:</b> {', '.join(themes)} <br>"
            html += f"<b>Rating:</b> {rating}"
            html += "</div>"
            
            display(HTML(html))
            
        except Exception as e:
            print(f"Error processing inputs: {str(e)}")

# Connect the logic to the button click event
button.on_click(on_button_click)

# Display the UI
display(fen_input, themes_input, rating_input, button, output)


Text(value='1k1r4/p1p4r/1p2R3/3P1qpp/2PQ1p2/1PN1n3/P5PP/4R2K b - - 5 25', description='FEN:', layout=Layout(wi…

Text(value='crushing,middlegame,short', description='Themes:', layout=Layout(width='80%'))

IntText(value=2076, description='Rating:')

Button(button_style='info', description='Predict Move', style=ButtonStyle())

Output()

In [79]:
dataset

,Unnamed: 0,FEN,Moves,Rating,Themes,Puzzle_FEN
0,0,r6k/pp2r2p/4Rp1Q/3p4/8/1N1P2R1/PqP2bPP/7K b - ...,f2g3 e6e7 b2b1 b3c1 b1c1 h6c1,1877,crushing hangingPiece long middlegame,r6k/pp2r2p/4Rp1Q/3p4/8/1N1P2b1/PqP3PP/7K w - -...
1,1,5rk1/1p3ppp/pq3b2/8/8/1P1Q1N2/P4PPP/3R2K1 w - ...,d3d6 f8d8 d6d8 f6d8,1501,advantage endgame short,5rk1/1p3ppp/pq1Q1b2/8/8/1P3N2/P4PPP/3R2K1 b - ...
2,2,8/4R3/1p2P3/p4r2/P6p/1P3Pk1/4K3/8 w - - 1 64,e7f7 f5e5 e2f1 e5e6,1355,advantage endgame rookEndgame short,8/5R2/1p2P3/p4r2/P6p/1P3Pk1/4K3/8 b - - 2 64
3,3,r2qr1k1/b1p2ppp/pp4n1/P1P1p3/4P1n1/B2P2Pb/3NBP...,b6c5 e2g4 h3g4 d1g4,1103,advantage middlegame short,r2qr1k1/b1p2ppp/p5n1/P1p1p3/4P1n1/B2P2Pb/3NBP1...
4,4,6k1/5p1p/4p3/4q3/3nN3/2Q3P1/PP3P1P/6K1 w - - 2 37,e4d2 d4e2 g1f1 e2c3,1422,crushing endgame fork short,6k1/5p1p/4p3/4q3/3n4/2Q3P1/PP1N1P1P/6K1 b - - ...
...,...,...,...,...,...,...
9995,9995,r2q1r1k/ppp3pp/5n2/b7/2Bp1NbQ/2P5/PP4PP/R1B2R1...,d4c3 f4g6,926,master mate mateIn1 oneMove opening pin,r2q1r1k/ppp3pp/5n2/b7/2B2NbQ/2p5/PP4PP/R1B2R1K...
9996,9996,1k1r1r2/2p5/Q1p4p/2P1qbp1/8/4p2P/P4PP1/1R3RK1 ...,f5b1 f1b1 e5b2 b1b2,945,endgame mate mateIn2 short,1k1r1r2/2p5/Q1p4p/2P1q1p1/8/4p2P/P4PP1/1b3RK1 ...
9997,9997,r1b2rk1/pp3pp1/4pb1p/8/2B1N2q/8/PP2QPPP/2R2RK1...,f6b2 g2g3 h4e7 e2b2,1767,advantage intermezzo middlegame short,r1b2rk1/pp3pp1/4p2p/8/2B1N2q/8/Pb2QPPP/2R2RK1 ...
9998,9998,2k2r2/p5Q1/Pppqp1p1/6N1/1bPP4/7P/1P4P1/6K1 b -...,d6f4 g7b7 c8d8 g5e6,1683,crushing endgame queensideAttack short,2k2r2/p5Q1/Ppp1p1p1/6N1/1bPP1q2/7P/1P4P1/6K1 w...


In [ ]:
n = 8
html = "<table><tr>"

dataset = pd.read_csv("./dataset/dataset.csv", nrows=10_000)

for i in range(n * 4):
    try:
        row = dataset.sample(1).iloc[0]
        fen_string = row["FEN"]
        # themes = row["Themes"].split(" ")
        # rating = row["Rating"]
        themes = []
        rating = 2000
        predicted_best_move = get_move(fen_string, themes, rating, steps=4, temperature=0.01)
        
        board = chess.Board(fen_string)
        arrows = [chess.svg.Arrow(chess.parse_square(predicted_best_move[:2]), chess.parse_square(predicted_best_move[2:4]), color='#0000ffcc')]
        
        svg = chess.svg.board(board, arrows=arrows, size=250)
        
        html += f"<td><div style='text-align: center; font-family: sans-serif;'>{svg}<br><b>Pred:</b> {predicted_best_move}</div></td>"
        
        if (i + 1) % 4 == 0 and i < (n * 4) - 1:
            html += "</tr><tr>"
    except:
        print("Error")

html += "</tr></table>"
display(HTML(html))


In [91]:
n = 8
html = "<table><tr>"

dataset = pd.read_csv("./dataset/dataset.csv", nrows=10_000)

for i in range(n * 4):
    try:
        row = dataset.sample(1).iloc[0]
        fen_string = row["FEN"]
        # themes = row["Themes"].split(" ")
        # rating = row["Rating"]
        themes = []
        rating = 2000
        predicted_best_move = get_move(fen_string, themes, rating, steps=4, temperature=0.01)
        
        board = chess.Board(fen_string)
        arrows = [chess.svg.Arrow(chess.parse_square(predicted_best_move[:2]), chess.parse_square(predicted_best_move[2:4]), color='#0000ffcc')]
        
        svg = chess.svg.board(board, arrows=arrows, size=250)
        
        html += f"<td><div style='text-align: center; font-family: sans-serif;'>{svg}<br><b>Pred:</b> {predicted_best_move}</div></td>"
        
        if (i + 1) % 4 == 0 and i < (n * 4) - 1:
            html += "</tr><tr>"
    except:
        print("Error")

html += "</tr></table>"
display(HTML(html))
